# Tendril notebook execution & training QA
Upload this `.ipynb` in **Explore → Run a notebook**, select an available lane, and run it. The default `smoke` mode uses synthetic data, bundled Python packages, and no external downloads. It exercises a complete training workflow, notebook features, and small downloadable artifacts.

**Expected default result:** Finished, `TENDRIL_QA_PASS` in the last cell, held-out metrics, inline PNG plots, and `results/qa_report.json`, model/checkpoint files, and `results/qa_bundle.zip`. CPU is sufficient; PyTorch/CUDA checks are opt-in.

Tendril limits: Python nbformat 4, 1.5 MB upload, 500 cells, 2 MB cell output, 4 MB total artifacts. Execution is capped by the server timeout (default 120 seconds, maximum 15 minutes) and prepaid credit. The smoke workload is deliberately small; actual time depends on hardware and startup.

Results appear after execution finishes, not as a live log stream. Download the executed notebook and artifacts promptly. The report covers checks inside the kernel; payment, polling, billing, and UI behavior require the manual checklist at the end.

In [ ]:
# Edit this cell BEFORE uploading. Run one QA mode per upload.
QA_MODE = 'smoke'  # smoke | training | runtime_error | output_limit | artifact_limit | timeout
RUN_TORCH = False  # Requires torch in a custom image; no automatic installation.
REQUIRE_CUDA = False  # Set with RUN_TORCH=True only when testing a GPU-capable image/lane.
SEED = 42

import os
for variable in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS', 'NUMEXPR_NUM_THREADS'):
    os.environ[variable] = '2'
assert QA_MODE in {'smoke', 'training', 'runtime_error', 'output_limit', 'artifact_limit', 'timeout'}
assert not REQUIRE_CUDA or RUN_TORCH, 'REQUIRE_CUDA also needs RUN_TORCH=True'
SAMPLES = 5000 if QA_MODE == 'training' else 1600
EPOCHS = 20 if QA_MODE == 'training' else 8
TREES = 60 if QA_MODE == 'training' else 24
print({'mode': QA_MODE, 'samples': SAMPLES, 'epochs': EPOCHS, 'torch': RUN_TORCH})

## 1. Environment, dependencies, and cross-cell state
Dependency failures should be visible immediately. Only non-secret runtime information is recorded. Files use the kernel working directory (`/work` on Tendril); this notebook also works in a writable local Jupyter directory.

In [ ]:
import asyncio, gzip, hashlib, importlib, importlib.metadata, json, logging
import multiprocessing, platform, sqlite3, subprocess, sys, threading, time, warnings, zipfile
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import psutil
import requests
from PIL import Image
from IPython.display import HTML, JSON, Markdown, display
from scipy import linalg, stats
from sklearn.datasets import make_classification, make_regression
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge, SGDClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, log_loss, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from threadpoolctl import threadpool_limits

threadpool_limits(limits=2)
STARTED = time.perf_counter()
rng = np.random.default_rng(SEED)
RESULTS = Path('results')
RESULTS.mkdir(exist_ok=True)
CHECKS = []
METRICS = {}
def record(name, details='', status='PASS'):
    CHECKS.append({'name': name, 'status': status, 'details': details})
    print(f'{status}: {name} — {details}', flush=True)

packages = ('nbformat', 'nbclient', 'ipykernel', 'numpy', 'pandas', 'matplotlib',
            'scipy', 'scikit-learn', 'pillow', 'psutil', 'requests', 'joblib')
ENVIRONMENT = {
    'python': platform.python_version(), 'platform': platform.platform(),
    'architecture': platform.machine(), 'cpu_count': os.cpu_count(),
    'working_directory': str(Path.cwd()),
    'packages': {name: importlib.metadata.version(name) for name in packages},
}
(RESULTS / 'environment.json').write_text(json.dumps(ENVIRONMENT, indent=2), encoding='utf-8')
record('runtime_dependencies', ENVIRONMENT['python'])
shared_state = {'seed': SEED, 'value': 21}
print(json.dumps(ENVIRONMENT, indent=2))

In [ ]:
assert shared_state['value'] == 21
shared_state['value'] *= 2
assert shared_state['value'] == 42
record('cross_cell_state', 'Variables and functions persisted from the previous cell')
# The final expression produces execute_result, rather than a print stream.
shared_state

## 2. IPython, shell commands, and output formats
`%pip` checks an already installed dependency with `--no-index`; no network access is needed. Expect stdout, stderr, a warning, plain-text rich-output fallbacks, and later PNG images. Tendril displays text and raster images; HTML/Markdown/JSON MIME data remains in the downloaded notebook.

In [ ]:
%pip --version
%pip install --no-index --no-deps psutil
# IPython shell magics report nonzero exits; assert them explicitly for QA.
assert get_ipython().user_ns.get('_exit_code') == 0, 'Offline %pip check failed'
%time sum(range(10000))
record('pip_and_line_magics', 'Offline dependency check and %time executed')

In [ ]:
!{sys.executable} -c "print('TENDRIL_SHELL_OK')"
assert get_ipython().user_ns.get('_exit_code') == 0
record('shell_escape', 'Python child process exited successfully')

In [ ]:
%%bash
set -eu
printf 'TENDRIL_BASH_OK\n'
printf 'created by bash cell\n' > results/shell.txt

In [ ]:
assert (RESULTS / 'shell.txt').read_text(encoding='utf-8') == 'created by bash cell\n'
record('bash_cell_magic', 'Shell artifact round trip verified')
print('stdout: training QA started — Unicode: café, 日本語, 🚀', flush=True)
print('stderr: intentional QA diagnostic, not a failed cell', file=sys.stderr, flush=True)
warnings.warn('Intentional QA warning: execution should continue', UserWarning)
logger = logging.getLogger('tendril.qa')
logger.handlers.clear()
logger.addHandler(logging.StreamHandler(sys.stdout))
logger.setLevel(logging.INFO)
logger.propagate = False
logger.info('logging: ready to train')
display(Markdown('**Rich Markdown output:** QA is running.'))
display(HTML('<strong>Rich HTML output:</strong> QA is running.'))
display(JSON({'qa': 'running', 'seed': SEED}))
record('stream_and_rich_outputs', 'Inspect stdout/stderr, Unicode, warning, and MIME data')

In [ ]:
%%capture captured_output
print('TENDRIL_CAPTURE_OK')

In [ ]:
assert captured_output.stdout.strip() == 'TENDRIL_CAPTURE_OK'
record('capture_cell_magic', 'Captured output persisted across cells')

## 3. NumPy/SciPy, tabular data, and storage
Generate data locally. Include missing values, then split before fitting preprocessing so held-out evaluation does not leak information. CSV, gzip, NPZ, and SQLite round trips check common training data workflows.

In [ ]:
matrix = rng.normal(size=(96, 96))
system = matrix.T @ matrix + np.eye(96)
target = rng.normal(size=96)
solution = linalg.solve(system, target, assume_a='pos')
residual = float(np.linalg.norm(system @ solution - target))
assert residual < 1e-8
assert np.isfinite(stats.norm.cdf(solution)).all()
METRICS['linear_solve_residual'] = residual
record('numpy_scipy', f'Linear solve residual={residual:.2e}')

In [ ]:
X, y = make_classification(n_samples=SAMPLES, n_features=12, n_informative=8,
                           n_redundant=2, n_clusters_per_class=1, class_sep=2.0,
                           flip_y=0.01, random_state=SEED)
columns = [f'feature_{index:02d}' for index in range(X.shape[1])]
frame = pd.DataFrame(X, columns=columns)
frame.loc[frame.index[::29], columns[0]] = np.nan
frame['label'] = y
csv_path = RESULTS / 'dataset.csv.gz'
frame.to_csv(csv_path, index=False, compression='gzip')
reloaded_frame = pd.read_csv(csv_path)
pd.testing.assert_frame_equal(frame, reloaded_frame, check_exact=False, rtol=1e-10, atol=1e-10)
X_train, X_test, y_train, y_test = train_test_split(
    frame[columns].to_numpy(), y, test_size=0.25, stratify=y, random_state=SEED)
preprocessor = make_pipeline(SimpleImputer(strategy='median'), StandardScaler())
train_scaled = preprocessor.fit_transform(X_train)
test_scaled = preprocessor.transform(X_test)
assert np.isfinite(train_scaled).all() and np.isfinite(test_scaled).all()
assert set(np.unique(y_train)) == {0, 1}
assert np.allclose(train_scaled.mean(axis=0), 0, atol=1e-10)
np.savez_compressed(RESULTS / 'data_sample.npz', X=train_scaled[:32], y=y_train[:32])
with np.load(RESULTS / 'data_sample.npz', allow_pickle=False) as stored:
    np.testing.assert_allclose(stored['X'], train_scaled[:32])
    np.testing.assert_array_equal(stored['y'], y_train[:32])
record('data_preprocessing_and_round_trips', f'train={len(y_train)}, held_out={len(y_test)}')
display(frame.head(5))

In [ ]:
database = sqlite3.connect(RESULTS / 'sample.sqlite')
try:
    frame.head(32).to_sql('samples', database, if_exists='replace', index=False)
    count = database.execute('SELECT COUNT(*) FROM samples').fetchone()[0]
    assert count == 32
    grouped = pd.read_sql_query('SELECT label, COUNT(*) AS count FROM samples GROUP BY label', database)
    assert int(grouped['count'].sum()) == 32
finally:
    database.close()
record('sqlite', '32 sample rows written and queried')
display(grouped)

## 4. Training, checkpoints, held-out evaluation, and inference
An incremental logistic classifier trains over epochs and saves a checkpoint halfway through. Each epoch logs loss and held-out accuracy. A resumed model must match uninterrupted training. A random forest and ridge regressor exercise additional training paths. Accuracy here is a synthetic-data QA threshold, not a production quality claim.

In [ ]:
classifier = SGDClassifier(loss='log_loss', learning_rate='constant', eta0=0.01,
                           alpha=0.0001, shuffle=False, random_state=SEED)
history = []
checkpoint_epoch = EPOCHS // 2
checkpoint_path = RESULTS / 'checkpoint.joblib'
training_started = time.perf_counter()
for epoch in range(1, EPOCHS + 1):
    classifier.partial_fit(train_scaled, y_train, classes=np.array([0, 1]))
    train_loss = float(log_loss(y_train, classifier.predict_proba(train_scaled)))
    held_out_accuracy = float(accuracy_score(y_test, classifier.predict(test_scaled)))
    assert np.isfinite(train_loss)
    history.append({'epoch': epoch, 'train_loss': train_loss, 'held_out_accuracy': held_out_accuracy})
    logger.info('epoch=%02d/%02d train_loss=%.4f held_out_accuracy=%.4f',
                epoch, EPOCHS, train_loss, held_out_accuracy)
    if epoch == checkpoint_epoch:
        joblib.dump({'epoch': epoch, 'model': classifier, 'preprocessor': preprocessor, 'seed': SEED},
                    checkpoint_path, compress=3)
history_frame = pd.DataFrame(history)
history_frame.to_csv(RESULTS / 'training_history.csv', index=False)
METRICS['classification_accuracy'] = history[-1]['held_out_accuracy']
METRICS['training_seconds'] = time.perf_counter() - training_started
assert METRICS['classification_accuracy'] >= 0.80, 'Held-out classification accuracy below QA threshold'
assert history[-1]['train_loss'] < history[0]['train_loss'], 'Training loss did not improve'
record('epoch_training', f"Held-out accuracy={METRICS['classification_accuracy']:.4f}")
display(history_frame)

In [ ]:
checkpoint = joblib.load(checkpoint_path)  # Load only this notebook's own artifacts.
assert checkpoint['epoch'] == checkpoint_epoch and checkpoint['seed'] == SEED
np.testing.assert_allclose(checkpoint['preprocessor'].transform(X_test), test_scaled)
resumed = checkpoint['model']
for epoch in range(checkpoint['epoch'] + 1, EPOCHS + 1):
    resumed.partial_fit(train_scaled, y_train, classes=np.array([0, 1]))
np.testing.assert_allclose(resumed.coef_, classifier.coef_, rtol=1e-10, atol=1e-10)
np.testing.assert_allclose(resumed.intercept_, classifier.intercept_, rtol=1e-10, atol=1e-10)
np.testing.assert_array_equal(resumed.predict(test_scaled), classifier.predict(test_scaled))
model_path = RESULTS / 'classifier.joblib'
joblib.dump({'model': classifier, 'preprocessor': preprocessor, 'features': columns}, model_path, compress=3)
loaded = joblib.load(model_path)
reloaded_predictions = loaded['model'].predict(loaded['preprocessor'].transform(X_test))
np.testing.assert_array_equal(reloaded_predictions, classifier.predict(test_scaled))
predictions_frame = pd.DataFrame({'actual': y_test[:20], 'predicted': reloaded_predictions[:20],
                                  'probability_1': classifier.predict_proba(test_scaled[:20])[:, 1]})
predictions_frame.to_csv(RESULTS / 'predictions.csv', index=False)
record('checkpoint_resume_and_inference', 'Resumed parameters and reloaded predictions match')
display(predictions_frame.head())

In [ ]:
forest = RandomForestClassifier(n_estimators=TREES, max_depth=8, n_jobs=2, random_state=SEED)
forest.fit(train_scaled, y_train)
METRICS['forest_accuracy'] = float(forest.score(test_scaled, y_test))
assert METRICS['forest_accuracy'] >= 0.80
regression_X, regression_y = make_regression(n_samples=600, n_features=8, noise=2.0, random_state=SEED)
reg_X_train, reg_X_test, reg_y_train, reg_y_test = train_test_split(
    regression_X, regression_y, test_size=0.25, random_state=SEED)
regressor = make_pipeline(StandardScaler(), Ridge(alpha=1.0))
regressor.fit(reg_X_train, reg_y_train)
reg_predictions = regressor.predict(reg_X_test)
METRICS['regression_r2'] = float(r2_score(reg_y_test, reg_predictions))
METRICS['regression_rmse'] = float(np.sqrt(mean_squared_error(reg_y_test, reg_predictions)))
assert METRICS['regression_r2'] >= 0.95
record('forest_and_regression', json.dumps({key: METRICS[key] for key in
       ('forest_accuracy', 'regression_r2', 'regression_rmse')}))

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(8, 3))
axes[0].plot(history_frame['epoch'], history_frame['train_loss'], marker='o')
axes[0].set(title='Training loss', xlabel='Epoch', ylabel='Log loss')
axes[1].plot(history_frame['epoch'], history_frame['held_out_accuracy'], marker='o')
axes[1].set(title='Held-out accuracy', xlabel='Epoch', ylabel='Accuracy', ylim=(0, 1))
figure.tight_layout()
figure.savefig(RESULTS / 'training_curves.png', dpi=100)
plt.show()
plt.close(figure)
confusion = confusion_matrix(y_test, reloaded_predictions)
figure, axis = plt.subplots(figsize=(3, 3))
axis.imshow(confusion, cmap='Blues')
axis.set(title='Held-out confusion matrix', xlabel='Predicted', ylabel='Actual', xticks=[0, 1], yticks=[0, 1])
for (row, column), value in np.ndenumerate(confusion):
    axis.text(column, row, str(value), ha='center', va='center', color='black')
figure.tight_layout()
figure.savefig(RESULTS / 'confusion_matrix.png', dpi=100)
plt.show()
plt.close(figure)
with Image.open(RESULTS / 'training_curves.png') as image:
    thumbnail = image.convert('RGB')
    thumbnail.thumbnail((320, 160))
    thumbnail.save(RESULTS / 'preview.jpg', quality=80)
    display(thumbnail)
record('plots_and_pillow', 'Two inline PNG plots and a JPEG thumbnail emitted')

## 5. Async, threads, child processes, and HTTP
The process-pool worker is an importable module so spawn-based multiprocessing works outside Linux/fork too. Support files live in a hidden directory and should not be returned as artifacts. HTTP uses a temporary loopback server; it does not contact external services.

In [ ]:
async def async_square(value):
    await asyncio.sleep(0.01)
    return value * value
async_results = await asyncio.gather(*(async_square(value) for value in range(8)))
assert async_results == [value * value for value in range(8)]
with ThreadPoolExecutor(max_workers=2) as pool:
    thread_results = list(pool.map(lambda value: value * value, range(8)))
assert thread_results == async_results
record('top_level_await_and_threads', str(async_results))

In [ ]:
support = Path('.tendril_qa_support')
support.mkdir(exist_ok=True)
(support / 'tendril_qa_worker.py').write_text(
    'def sum_squares(limit):\n    return sum(value * value for value in range(limit))\n', encoding='utf-8')
sys.path.insert(0, str(support.resolve()))
importlib.invalidate_caches()
from tendril_qa_worker import sum_squares
limits = [100, 200, 300, 400]
with ProcessPoolExecutor(max_workers=2, mp_context=multiprocessing.get_context('spawn')) as pool:
    process_results = list(pool.map(sum_squares, limits))
assert process_results == [sum(value * value for value in range(limit)) for limit in limits]
record('spawn_process_pool', str(process_results))

In [ ]:
child = subprocess.run([sys.executable, '-c',
    "import sys; print('child stdout'); print('child stderr', file=sys.stderr)"],
    capture_output=True, text=True, timeout=10, check=True)
assert child.stdout.strip() == 'child stdout' and child.stderr.strip() == 'child stderr'
expected_failure = subprocess.run([sys.executable, '-c', 'import sys; sys.exit(3)'],
                                  capture_output=True, text=True, timeout=10)
assert expected_failure.returncode == 3
try:
    raise ValueError('Expected, handled validation failure')
except ValueError as error:
    print('Handled error:', error)
record('subprocess_and_handled_errors', 'stdout/stderr, exit code, and caught exception verified')

In [ ]:
class QAHandler(BaseHTTPRequestHandler):
    def do_GET(self):
        payload = json.dumps({'ok': True, 'rows': SAMPLES}).encode('utf-8')
        self.send_response(200)
        self.send_header('Content-Type', 'application/json')
        self.send_header('Content-Length', str(len(payload)))
        self.end_headers()
        self.wfile.write(payload)
    def log_message(self, *args):
        pass

server = ThreadingHTTPServer(('127.0.0.1', 0), QAHandler)
server_thread = threading.Thread(target=server.serve_forever, daemon=True)
server_thread.start()
try:
    with requests.Session() as session:
        session.trust_env = False  # Keep loopback traffic independent of proxy configuration.
        response = session.get(f'http://127.0.0.1:{server.server_port}/qa', timeout=5)
        response.raise_for_status()
        assert response.json() == {'ok': True, 'rows': SAMPLES}
finally:
    server.shutdown()
    server.server_close()
    server_thread.join(timeout=5)
assert not server_thread.is_alive()
record('requests_loopback_http', 'Local JSON request succeeded; server cleaned up')

## 6. Optional PyTorch/CUDA training
Leave `RUN_TORCH=False` for bundled CPU images: they do not include PyTorch. To test a custom PyTorch image, enable it in the configuration cell. `REQUIRE_CUDA=True` fails clearly if CUDA is unavailable. Priority and E2B notebook lanes are CPU-only. No package or model download happens here.

In [ ]:
if not RUN_TORCH:
    record('torch_training', 'Disabled; bundled CPU images do not require torch', status='SKIP')
else:
    import torch
    torch.set_num_threads(2)
    torch.manual_seed(SEED)
    cuda_available = torch.cuda.is_available()
    assert not REQUIRE_CUDA or cuda_available, 'CUDA required but unavailable'
    device = torch.device('cuda' if cuda_available else 'cpu')
    if cuda_available:
        torch.cuda.manual_seed_all(SEED)
    neural_model = torch.nn.Sequential(torch.nn.Linear(12, 16), torch.nn.ReLU(), torch.nn.Linear(16, 2)).to(device)
    optimizer = torch.optim.Adam(neural_model.parameters(), lr=0.03)
    neural_X = torch.tensor(train_scaled[:512], dtype=torch.float32, device=device)
    neural_y = torch.tensor(y_train[:512], dtype=torch.long, device=device)
    criterion = torch.nn.CrossEntropyLoss()
    neural_losses = []
    neural_model.train()
    for step in range(30):
        optimizer.zero_grad()
        loss = criterion(neural_model(neural_X), neural_y)
        assert torch.isfinite(loss).item()
        loss.backward()
        optimizer.step()
        neural_losses.append(float(loss.detach().cpu()))
        if step % 10 == 0:
            print(f'torch step={step:02d} loss={neural_losses[-1]:.4f} device={device}', flush=True)
    assert neural_losses[-1] < neural_losses[0]
    neural_model.eval()
    with torch.no_grad():
        neural_test = torch.tensor(test_scaled, dtype=torch.float32, device=device)
        neural_predictions = neural_model(neural_test).argmax(dim=1).cpu().numpy()
    METRICS['torch_accuracy'] = float(accuracy_score(y_test, neural_predictions))
    assert METRICS['torch_accuracy'] >= 0.80
    torch_path = RESULTS / 'torch_checkpoint.pt'
    torch.save({name: tensor.detach().cpu() for name, tensor in neural_model.state_dict().items()}, torch_path)
    restored_neural_model = torch.nn.Sequential(torch.nn.Linear(12, 16), torch.nn.ReLU(), torch.nn.Linear(16, 2))
    restored_neural_model.load_state_dict(torch.load(torch_path, map_location='cpu', weights_only=True))
    restored_neural_model.eval()
    with torch.no_grad():
        restored_predictions = restored_neural_model(torch.tensor(test_scaled, dtype=torch.float32)).argmax(dim=1).numpy()
    np.testing.assert_array_equal(restored_predictions, neural_predictions)
    ENVIRONMENT['torch'] = {'version': torch.__version__, 'device': str(device), 'cuda_available': cuda_available}
    record('torch_training', f"device={device}, accuracy={METRICS['torch_accuracy']:.4f}; reload matched")

## 7. Artifact integrity, report, and download bundle
The default artifact set stays under a conservative 3.5 MB budget, including the ZIP. `artifact_manifest.json` records sizes and SHA-256 hashes for all preceding result files; the manifest and bundle do not hash themselves. A hidden sentinel should be absent from Tendril downloads. `qa_report.json` reports completed checks before any deliberately failing mode runs.

In [ ]:
unicode_path = RESULTS / 'unicode sample.txt'
unicode_text = 'Tendril QA — café, 日本語, 🚀\n'
unicode_path.write_text(unicode_text, encoding='utf-8')
assert unicode_path.read_text(encoding='utf-8') == unicode_text
binary_data = bytes(range(256)) * 16
binary_path = RESULTS / 'sample.bin'
binary_path.write_bytes(binary_data)
assert binary_path.read_bytes() == binary_data
with gzip.open(RESULTS / 'sample.txt.gz', 'wt', encoding='utf-8') as compressed:
    compressed.write(unicode_text)
with gzip.open(RESULTS / 'sample.txt.gz', 'rt', encoding='utf-8') as compressed:
    assert compressed.read() == unicode_text
(RESULTS / '.hidden_sentinel.txt').write_text('This file must not be returned by Tendril.', encoding='utf-8')
record('text_binary_and_compression', 'UTF-8, space in filename, binary bytes, and gzip verified')
METRICS['process_rss_mb'] = round(psutil.Process().memory_info().rss / 1_000_000, 2)
METRICS['elapsed_seconds_before_bundle'] = round(time.perf_counter() - STARTED, 3)
print('Kernel RSS MB:', METRICS['process_rss_mb'])

In [ ]:
REPORT = {
    'schema_version': 1, 'mode': QA_MODE, 'seed': SEED,
    'status': 'PASS' if QA_MODE in {'smoke', 'training', 'artifact_limit'} else 'EXPECTED_FAILURE_PENDING',
    'checks': CHECKS, 'metrics': METRICS, 'environment': ENVIRONMENT,
    'note': 'Kernel checks only. Verify upload, polling, billing, rendering, and downloads separately.',
}
report_path = RESULTS / 'qa_report.json'
report_path.write_text(json.dumps(REPORT, indent=2, ensure_ascii=False, allow_nan=False), encoding='utf-8')
assert json.loads(report_path.read_text(encoding='utf-8'))['checks'] == CHECKS
manifest = []
for path in sorted(RESULTS.iterdir()):
    if path.is_file() and not path.name.startswith('.') and path.name not in {'artifact_manifest.json', 'qa_bundle.zip', 'oversized.bin'}:
        data = path.read_bytes()
        manifest.append({'name': path.as_posix(), 'bytes': len(data), 'sha256': hashlib.sha256(data).hexdigest()})
manifest_path = RESULTS / 'artifact_manifest.json'
manifest_path.write_text(json.dumps(manifest, indent=2), encoding='utf-8')
bundle_path = RESULTS / 'qa_bundle.zip'
with zipfile.ZipFile(bundle_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for item in manifest:
        archive.write(item['name'], arcname=Path(item['name']).name)
    archive.write(manifest_path, arcname=manifest_path.name)
with zipfile.ZipFile(bundle_path) as archive:
    assert archive.testzip() is None
    for item in manifest:
        assert hashlib.sha256(archive.read(Path(item['name']).name)).hexdigest() == item['sha256']
artifact_bytes = sum(item['bytes'] for item in manifest) + manifest_path.stat().st_size + bundle_path.stat().st_size
assert artifact_bytes < 3_500_000, f'QA artifacts too large: {artifact_bytes} bytes'
print(f'Artifacts: {len(manifest) + 2} visible files, {artifact_bytes:,} bytes including ZIP')
print('Artifact hashes and ZIP integrity verified.')
display(pd.DataFrame(CHECKS))

## 8. Opt-in failure and limit tests
Change `QA_MODE` in the first cell and upload again. Run each mode separately; an uncaught failure stops subsequent cells.

| Mode | Expected Tendril result | What to inspect |
| --- | --- | --- |
| `smoke` | Finished | All default checks pass; final success marker and small downloads |
| `training` | Finished | More rows, epochs, and trees; same QA checks |
| `runtime_error` | Failed | Intentional ValueError; earlier logs, plots, report, and bundle preserved; no final success marker |
| `output_limit` | Failed | Runner reports output exceeded 2 MB; no final success marker |
| `artifact_limit` | Finished with runner notice | `oversized.bin` is skipped; normal small artifacts remain downloadable |
| `timeout` | Failed | Deadline stops the sleeping cell; no final success marker. Partial results may be unavailable after sandbox termination |

`timeout` deliberately uses the remaining execution budget. For a quick test, configure a short server `RUN_TIMEOUT_MS` that still permits the earlier QA cells to finish. Failure modes consume execution time too. Invalid syntax is tested separately in the manual checklist because Tendril preflights every code cell before running anything.

In [ ]:
if QA_MODE == 'runtime_error':
    print('TENDRIL_QA_EXPECTED_RUNTIME_ERROR: earlier outputs and files should be retained.', flush=True)
    raise ValueError('TENDRIL_QA_INTENTIONAL_FAILURE: testing partial notebook results')
elif QA_MODE == 'output_limit':
    print('TENDRIL_QA_EXPECTED_OUTPUT_LIMIT', flush=True)
    print('Q' * 2_100_000, flush=True)
    raise AssertionError('Output cap did not stop oversized output')
elif QA_MODE == 'artifact_limit':
    oversized_path = RESULTS / 'oversized.bin'
    with oversized_path.open('wb') as oversized:
        oversized.truncate(4_000_001)
    assert oversized_path.stat().st_size > 4_000_000
    print('TENDRIL_QA_EXPECTED_ARTIFACT_SKIP: oversized.bin must not be downloadable.', flush=True)
elif QA_MODE == 'timeout':
    print('TENDRIL_QA_EXPECTED_TIMEOUT: sleeping beyond the maximum notebook deadline.', flush=True)
    time.sleep(1000)
    raise AssertionError('Execution deadline did not stop sleeping cell')
else:
    print('Negative tests disabled; normal successful run.')

In [ ]:
assert QA_MODE in {'smoke', 'training', 'artifact_limit'}, 'A failing QA mode reached the success cell'
assert all(check['status'] in {'PASS', 'SKIP'} for check in CHECKS)
assert shared_state['value'] == 42
assert model_path.is_file() and checkpoint_path.is_file() and bundle_path.is_file()
print('TENDRIL_QA_PASS', flush=True)
print(f"Mode={QA_MODE}; passed={sum(check['status'] == 'PASS' for check in CHECKS)}; "
      f"skipped={sum(check['status'] == 'SKIP' for check in CHECKS)}; elapsed={time.perf_counter() - STARTED:.2f}s")
print(json.dumps(METRICS, indent=2, allow_nan=False))
print('Download qa_report.json, qa_bundle.zip, and the executed notebook.')

## 9. Manual end-to-end QA checklist

- **Upload:** file picker and drag/drop accept this notebook and show correct total/code cell counts. Try each available contributor, priority, and E2B lane independently.
- **Payment and polling:** wallet approval proceeds to provisioning/running; results arrive after completion. With the API, keep polling using `jobId`/`jobToken` until `run` or `error` exists, even if status is already terminal. Inspect billed seconds/cost and refreshed balance.
- **Rendering:** epoch logs, stdout/stderr, Unicode, warnings, tables as plain text, two PNG plots, and a JPEG appear. Rich HTML/Markdown/JSON is available in the executed notebook; the web results view uses text/raster fallbacks.
- **Downloads:** executed notebook opens in Jupyter with sources, execution counts, and outputs. Extract `qa_bundle.zip`; verify file hashes against `artifact_manifest.json`. UTF-8/spaced filenames and binary files download intact. Hidden sentinel/support files are absent. Model/checkpoint reload checks already ran inside this notebook.
- **Failure paths:** upload each opt-in mode separately and compare to the table above. Earlier cells survive runtime errors when result collection succeeds; later success cell must remain unexecuted. Oversized artifact mode retains normal downloads and emits a skip notice.
- **Syntax preflight:** on a copy, append a code cell with `if True print('bad syntax')`. Expect SyntaxError before any training cells execute; source stays unchanged.
- **Upload rejection:** on separate copies, use an R kernelspec/language, nbformat 3, 501 cells, or a file above 1.5 MB. Also upload malformed JSON. Expect validation before payment.
- **Concurrency/budget:** try a second notebook with the same payer while a job or session is active; expect `payer_busy`. Test insufficient credit and a depleted execution budget using the configured test environment.
- **Repeatability:** rerun `smoke` with the same seed. Classification/regression metrics and checkpoint equivalence should match within numerical tolerance; runtimes and archive hashes need not match.

The notebook does not assert remote job billing, UI rendering, retention/eviction, external internet connectivity, or availability of GPU hardware. Those need their own end-to-end checks.